# <center> Deep Learning Coursework </center>
## <center> Audio Text Transformer Implementation </center>

Explaination of Audio Text Transformer

Imports required

In [ ]:
import torch
import torch.backends.cudnn
import numpy as np
from torch import nn, optim
from torch.nn import functional as F
import torchvision.datasets
from torch.optim.optimizer import Optimizer
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision import transforms

In [ ]:
%run audio-text_transformer.py --learning-rate 1e-4 --seed 42 --optimizer adam --model-name baseline --weights-path transformer_weights.pth

In [ ]:
weights = torch.load(
    "transformer_weights.pth",
    map_location="cpu",
    weights_only=True
)

for name, tensor in weights.items():
    print(f"Layer: {name}")
    print(f"Shape: {tensor.shape}")
    print(f"Weights:\n{tensor}")
    print("-" * 50)

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torch import nn

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_root = Path("/data/COMS30017")

# Load validation features
val_audio, val_text, val_labels = load_features(
    dataset_root, "val"
)

# Create validation dataset
val_dataset = AudioTextDataset(
    val_audio, val_text, val_labels, training=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

# Get feature dimensions from the dataset
first_audio, first_text, _ = val_dataset[0]

# Initialise your actual model
model = AudioTextTransformer(
    audio_dim=first_audio.shape[-1],
    text_dim=first_text.shape[-1],
    d_model=MODEL_DIM,
    num_heads=4,
    mlp_dim=TRANSFORMER_MLP_DIM,
    dropout=DROPOUT
)

# Load trained model weights
model.load_state_dict(
    torch.load(
        "transformer_weights.pth",
        map_location=DEVICE,
        weights_only=True
    )
)

model = model.to(DEVICE)

# Evaluation setup
criterion = nn.CrossEntropyLoss()
model.eval()

total_loss = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for audio, text, labels in val_loader:

        audio = audio.to(DEVICE)
        text = text.to(DEVICE)
        labels = labels.to(DEVICE)

        # Get model predictions
        logits = model(audio, text)

        # Calculate loss
        loss = criterion(logits, labels)
        total_loss += loss.item()

        # Get predicted classes
        predictions = logits.argmax(dim=-1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
accuracy = np.mean(
    np.array(all_preds) == np.array(all_labels)
)

average_loss = total_loss / len(val_loader)

print(f"Validation Loss: {average_loss:.5f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")
print(f"Total Samples: {len(all_labels)}")

Extension: AdamW vs Adam (accuracy and sustainability comparison)

In [ ]:
%run audio-text_transformer.py --learning-rate 1e-4 --seed 42 --optimizer adamw --weight-decay 1e-2 --model-name adamw --weights-path adamw_weights.pth

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torch import nn

# AdamW model evaluation (same as the baseline evaluation above, different weights file)
# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_root = Path("/data/COMS30017")

# Load validation features
val_audio, val_text, val_labels = load_features(
    dataset_root, "val"
)

# Create validation dataset
val_dataset = AudioTextDataset(
    val_audio, val_text, val_labels, training=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

# Get feature dimensions from the dataset
first_audio, first_text, _ = val_dataset[0]

# Initialise your actual model
model = AudioTextTransformer(
    audio_dim=first_audio.shape[-1],
    text_dim=first_text.shape[-1],
    d_model=MODEL_DIM,
    num_heads=4,
    mlp_dim=TRANSFORMER_MLP_DIM,
    dropout=DROPOUT
)

# Load trained model weights
model.load_state_dict(
    torch.load(
        "adamw_weights.pth",
        map_location=DEVICE,
        weights_only=True
    )
)

model = model.to(DEVICE)

# Evaluation setup
criterion = nn.CrossEntropyLoss()
model.eval()

total_loss = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for audio, text, labels in val_loader:

        audio = audio.to(DEVICE)
        text = text.to(DEVICE)
        labels = labels.to(DEVICE)

        # Get model predictions
        logits = model(audio, text)

        # Calculate loss
        loss = criterion(logits, labels)
        total_loss += loss.item()

        # Get predicted classes
        predictions = logits.argmax(dim=-1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
accuracy = np.mean(
    np.array(all_preds) == np.array(all_labels)
)

average_loss = total_loss / len(val_loader)

print(f"Validation Loss: {average_loss:.5f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")
print(f"Total Samples: {len(all_labels)}")

In [ ]:
import json
import matplotlib.pyplot as plt

runs = {}
for name in ["baseline", "adamw"]:
    with open(f"{name}/metrics.json") as f:
        runs[name] = json.load(f)

rows = [
    ("Optimiser",                   lambda m: m["optimizer"]),
    ("Weight decay",                lambda m: m["weight_decay"]),
    ("Trainable parameters",        lambda m: f"{m['parameters']:,}"),
    ("Best val accuracy (%)",       lambda m: f"{m['best_val_accuracy'] * 100:.2f}"),
    ("Best val epoch",              lambda m: m["best_val_epoch"]),
    ("Final val accuracy (%)",      lambda m: f"{m['final_val_accuracy'] * 100:.2f}"),
    ("Final val loss",              lambda m: f"{m['final_val_loss']:.4f}"),
    ("Epochs to reach target val acc", lambda m: m["epochs_to_target"]),
    ("Total train time (s)",        lambda m: f"{m['train_time_s']:.1f}"),
    ("Time per epoch (s)",          lambda m: f"{m['time_per_epoch_s']:.2f}"),
    ("Time to target (s)",          lambda m: (f"{m['epochs_to_target'] * m['time_per_epoch_s']:.1f}"
                                               if m["epochs_to_target"] else "not reached")),
    ("Peak GPU memory (MB)",        lambda m: (f"{m['peak_gpu_mem_mb']:.0f}"
                                               if m["peak_gpu_mem_mb"] is not None else "n/a (CPU)")),
]

print(f"{'Metric':<34}{'Adam (baseline)':>18}{'AdamW':>18}")
print("-" * 70)
for label, fn in rows:
    print(f"{label:<34}{str(fn(runs['baseline'])):>18}{str(fn(runs['adamw'])):>18}")

# Validation accuracy / loss curves for both optimisers
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, label in [("baseline", "Adam"), ("adamw", "AdamW")]:
    h = runs[name]["history"]
    axes[0].plot(h["val_epochs"], [a * 100 for a in h["val_accuracy"]], marker="o", label=label)
    axes[1].plot(h["val_epochs"], h["val_loss"], marker="o", label=label)
axes[0].axhline(runs["baseline"]["target_accuracy"] * 100, color="grey", linestyle="--", label="Target")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Validation accuracy (%)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation loss")
for ax in axes:
    ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig("adam_vs_adamw.png", dpi=300, bbox_inches="tight")
plt.show()